##### Use this notebook to run a default T2f test including benchmarks vs competing model

In [1]:
%load_ext autoreload

In [2]:
%autoreload 2

In [3]:
!pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable
  Using cached pip-26.2.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.2-py3-none-any.whl (1.8 MB)


ERROR: To modify pip, please run the following command:
C:\Python314\python.exe -m pip install --upgrade pip


In [4]:
!pip install -r requirements.txt --quiet

  error: subprocess-exited-with-error
  
  × installing build dependencies for matplotlib did not run successfully.
  │ exit code: 1
  ╰─> [52 lines of output]
        Using cached meson_python-0.22.1-py3-none-any.whl.metadata (2.1 kB)
        Using cached pybind11-3.1.0-py3-none-any.whl.metadata (10 kB)
        Using cached setuptools_scm-10.3.4-py3-none-any.whl.metadata (6.7 kB)
        Using cached numpy-2.2.6.tar.gz (20.3 MB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
        Installing backend dependencies: started
        Installing backend dependencies: finished with status 'done'
        Preparing metadata (pyproject.toml): started
        Preparing metadata (pyproject.toml): finished with status 'error'
        error: subprocess-exited-with-error
      
        Ã— Preparing metadata 

In [5]:
import numpy as np

In [6]:
import sys
import os
sys.path.insert(0, os.path.abspath('t2f/'))

In [7]:
from t2f.t2f_base import data_load, t2f_fit, T2f

In [8]:
from benchmarks.helpers import evaluate_forecast, smape, wsmae, _metrics, \
                save_benchmark_results, create_benchmark_plots

In [9]:
sys.path.insert(0, os.path.abspath('benchmarks/'))

In [10]:
from benchmarks.bench_base import run_bench

c:\Users\Jenny\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [11]:
ep = 2
dataset='eld'
input_data_path='./datasets/eld/diff_eld.parquet'

series_id_col='id'                      ## column keeping the local series index
timestamp_col='timestamp'               ## column with relevant timestamp
timestamp_format='%Y-%m-%d %H:%M:%S'    ## string format for timestamp_col
covariate_cols=['tmax', 'tmin', 'tmed'] ## covariates to consider

weight_col='cost_kwh'                   ## objective weight variable (can be part of the prediction covariates)
target_col='load'                       ## prediction variable

local_length=2                          ## Controls the output sequence length from local stage
global_length=3                         ## Controls the output sequence length from global stage

test_size=0.2
batch_size=16

In [12]:
t2f_refit_ = False

In [14]:
## get data
train_df, test_df, prepared_df, freq = \
    data_load(
        input_data_path,
        series_id_col,
        timestamp_col,
        timestamp_format,
        covariate_cols,
        weight_col,
        target_col,
        test_size,
        )

Preload data length: 10999718
Preload ids: 326
Loaded data length: 10999718
Loaded ids: 326


In [15]:
## fit T2f
if t2f_refit_:
    t2f_ = t2f_fit(
            train_df=train_df,
            test_df=test_df,
            prepared_df=prepared_df,
            freq=freq,
            series_id_col=series_id_col,
            timestamp_col=timestamp_col,
            covariate_cols=covariate_cols,
            weight_col=weight_col,
            target_col=target_col,
            local_length=local_length,
            global_length=global_length,
            epochs=ep,
            batch_size=batch_size,
            model_save_path=f'./t2f/model-store/{dataset}/l{local_length}_g{global_length}/ep{ep}'
    )

In [16]:
t2f_load = T2f.load(f'./t2f/model-store/{dataset}/l{local_length}_g{global_length}/ep{ep}')

T2fError: Failed to load T2f model: [Errno 2] No such file or directory: './t2f/model-store/eld/l2_g3/ep2\\attributes.pkl'

In [ ]:
X_test = test_df[covariate_cols]
t2f_predictions = t2f_load.predict(X_test)

In [ ]:
t2f_load.batch_size

In [ ]:
t2f_load.logger

In [ ]:
## models to be included for testing

models = ['AUTOARIMA', 
          'GARCH', 
          'N-BEATS', 
          'Prophet', 
          'GRU', 
          'TCN', 
          'DeepAR local', 
          'DeepAR global'
         ]

In [ ]:
local_results, global_results = run_bench(
        t2f_predictions=t2f_predictions.reset_index(),
        train_df=train_df,
        test_df=test_df,
        hidden_size=t2f_load.hidden_size, #n-beats, gru, deepar
        num_layers=t2f_load.num_layers, #gru, deepar
        epochs=ep, #n-beats, gru, tcn, deepar
        batch_size=t2f_load.batch_size, #n-beats, gru, tcn
        logger=t2f_load.logger,
        series_id_col=series_id_col,
        timestamp_col=timestamp_col,
        covariate_cols=covariate_cols,
        weight_col=weight_col,
        target_col=target_col,
        local_length=local_length,
        global_length=global_length,
        bench_models=models
        )

In [ ]:
avg_local_results, global_metrics = _metrics(['T2f'] + models, 
                                             local_results, global_results, 
                                             exclude_metrics = [] #mae, rmse, wsmae, smape
                                                )

In [ ]:
print("Local Results:")
print(avg_local_results.head())
print("\nLocal Results columns:", avg_local_results.columns.tolist())

print("\nGlobal Metrics:")
print(global_metrics.head())
print("\nGlobal Metrics columns:", global_metrics.columns.tolist())

In [ ]:
# save
save_benchmark_results(
    avg_local_results, 
    global_metrics,
    filepath=f'./benchmarks/{dataset}/l{local_length}_g{global_length}/ep{ep}/results.json',
    dataset=dataset,
    epochs=ep,
    local_length=local_length,
    global_length=global_length,
    test_size=test_size,
    batch_size=batch_size
)

In [ ]:
# Define parameters
metrics = ['smape', 'mae', 'rmse', 'wsmae']
plot_types = ['box', 'rank', 'radar', 'local_bars', 'local_lines', 
              'global_bars', 'global_lines']

create_benchmark_plots(
    local_results=avg_local_results,
    global_metrics=global_metrics,
    metrics=metrics,
    models=['T2f'] + models,
    plot_types=plot_types,
    save_plots=True,
    save_dir=f'./benchmarks/{dataset}/l{local_length}_g{global_length}/ep{ep}/benchmark_plots/'
)

In [ ]:
for ep in range(5,55,5):
    print(ep)

    t2f_load = T2f.load(f'./t2f/model-store/{dataset}/l{local_length}_g{global_length}/ep{ep}')

    X_test = test_df[covariate_cols]
    t2f_predictions = t2f_load.predict(X_test)

    models = ['AUTOARIMA', 'GARCH', 'N-BEATS', 'Prophet', 'GRU', 'TCN', 
              'DeepAR local', 
              'DeepAR global'
             ]

    local_results, global_results = run_bench(
            t2f_predictions=t2f_predictions.reset_index(),
            train_df=train_df,
            test_df=test_df,
            hidden_size=t2f_load.hidden_size, #n-beats, gru, deepar
            num_layers=t2f_load.num_layers, #gru, deepar
            epochs=ep, #n-beats, gru, tcn, deepar
            batch_size=t2f_load.batch_size, #n-beats, gru, tcn
            logger=t2f_load.logger,
            series_id_col=series_id_col,
            timestamp_col=timestamp_col,
            covariate_cols=covariate_cols,
            weight_col=weight_col,
            target_col=target_col,
            local_length=local_length,
            global_length=global_length,
            bench_models=models
            )

    avg_local_results, global_metrics = _metrics(['T2f'] + models, 
                                                 local_results, global_results, 
                                                 exclude_metrics = [] #mae, rmse, wsmae, smape
                                                    )
    # save
    save_benchmark_results(
        avg_local_results, 
        global_metrics,
        filepath=f'./benchmarks/{dataset}/l{local_length}_g{global_length}/ep{ep}/results.json',
        dataset=dataset,
        epochs=ep,
        local_length=local_length,
        global_length=global_length,
        test_size=test_size,
        batch_size=batch_size
    )

    # Define parameters
    metrics = ['smape', 'mae', 'rmse', 'wsmae']
    plot_types = ['box', 'rank', 'radar', 'local_bars', 'local_lines', 
                  'global_bars', 'global_lines']

    create_benchmark_plots(
        local_results=avg_local_results,
        global_metrics=global_metrics,
        metrics=metrics,
        models=['T2f'] + models,
        plot_types=plot_types,
        save_plots=True,
        save_dir=f'./benchmarks/{dataset}/l{local_length}_g{global_length}/ep{ep}/benchmark_plots/'
    )
    